# Calibration-ready GBDT biomarker experts
This separate notebook preserves the original experiments. It trains one source-specific model per target, with 70% development, 15% calibration, and 15% untouched test partitions created independently inside each cohort. Development-only stratified CV selects the estimator. Calibration predictions are exported raw for the selected-model calibration notebook. UCI time is excluded. Multiclass MI cause codes are not treated as binary outcomes.

## 1. Imports and data locations

In [1]:
from pathlib import Path
import json, warnings, joblib, hashlib
warnings.filterwarnings('ignore')
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier,StackingClassifier
from sklearn.linear_model import LogisticRegression
from src.training.biomarker_training_data import load_biomarker_source
root=Path.cwd().resolve()
while not (root/'src').is_dir():
    if root.parent==root: raise FileNotFoundError('Open from ml-service')
    root=root.parent
DATA_ROOT=root/'data/raw/biomarkers'
OUT=root/'artifacts/03_training/biomarkers/selected_gbdt_stacking/v1'
OUT.mkdir(parents=True,exist_ok=True)
SEED=42;N_FOLDS=5;MIN_CLASS_SUPPORT=6
DATASETS=['zheen','uci_heart_failure','framingham','mi_complications']
def file_sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()


## 2. Candidate GBDT, stacking and baseline models

In [2]:
def candidates(y):
    counts=pd.Series(y).value_counts()
    weight=float(counts.min()/counts.max()) if len(counts)==2 else 1.0
    models={
      'LogisticRegression':LogisticRegression(max_iter=1000,class_weight='balanced'),
      'RandomForest':RandomForestClassifier(n_estimators=300,max_depth=8,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1),
      'HistGradientBoosting':HistGradientBoostingClassifier(max_iter=200,max_leaf_nodes=15,learning_rate=.05,l2_regularization=1.0,random_state=SEED),
    }
    bases=[]
    try:
      from xgboost import XGBClassifier
      xgb=XGBClassifier(n_estimators=200,max_depth=4,learning_rate=.05,subsample=.8,colsample_bytree=.8,
                        eval_metric='logloss',random_state=SEED,n_jobs=-1,scale_pos_weight=1/max(weight,1e-3))
      models['XGBoost']=xgb;bases.append(('xgb',xgb))
    except ImportError: pass
    try:
      from lightgbm import LGBMClassifier
      lgbm=LGBMClassifier(n_estimators=200,max_depth=5,learning_rate=.05,class_weight='balanced',random_state=SEED,verbosity=-1)
      models['LightGBM']=lgbm;bases.append(('lgbm',lgbm))
    except ImportError: pass
    try:
      from catboost import CatBoostClassifier
      cat=CatBoostClassifier(iterations=250,depth=5,learning_rate=.05,auto_class_weights='Balanced',random_seed=SEED,verbose=False)
      models['CatBoost']=cat;bases.append(('cat',cat))
    except ImportError: pass
    if len(bases)>=2:
      models['Stacked_GBDT']=StackingClassifier(estimators=bases,final_estimator=LogisticRegression(max_iter=1000),
                                                  cv=3,stack_method='predict_proba',n_jobs=1)
    return models


## 3. Independent development selection, calibration/test predictions, and split manifest

In [3]:
split_rows=[];cv_rows=[];prediction_rows=[]
for source in DATASETS:
    data=load_biomarker_source(DATA_ROOT,source)
    for target_name,target_series in data.targets.items():
        if data.target_specs[target_name]['type']!='binary':
            print(f'[Skipped calibration] {source}/{target_name}: endpoint is multiclass')
            continue
        y=pd.to_numeric(target_series,errors='coerce')
        valid=y.notna().to_numpy();X=data.features.loc[valid].reset_index(drop=True);y=y.loc[valid].astype(int).reset_index(drop=True)
        if y.nunique()!=2 or int(y.value_counts().min())<MIN_CLASS_SUPPORT:
            print(f'[Skipped] {source}/{target_name}: inadequate binary support');continue
        ids=np.asarray([f'{source}:{i}' for i in np.flatnonzero(valid)])
        idx=np.arange(len(y));dev,held=train_test_split(idx,test_size=.30,random_state=SEED,stratify=y)
        cal,test=train_test_split(held,test_size=.50,random_state=SEED,stratify=y.iloc[held])
        folds=min(N_FOLDS,int(y.iloc[dev].value_counts().min()))
        cv=StratifiedKFold(n_splits=folds,shuffle=True,random_state=SEED)
        best_name=None;best_score=-np.inf
        for name,est in candidates(y.iloc[dev]).items():
            pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',clone(est))])
            try:
                score=float(np.mean(cross_val_score(pipe,X.iloc[dev],y.iloc[dev],cv=cv,scoring='roc_auc',n_jobs=1)))
                cv_rows.append({'dataset':source,'target':target_name,'model':name,'development_cv_roc_auc':score,'folds':folds})
                if np.isfinite(score) and score>best_score:best_name,best_score=name,score
            except (ValueError,TypeError) as exc:cv_rows.append({'dataset':source,'target':target_name,'model':name,'error':str(exc)[:250]})
        if best_name is None:raise RuntimeError(f'No model selected for {source}/{target_name}')
        model=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',clone(candidates(y.iloc[dev])[best_name]))])
        model.fit(X.iloc[dev],y.iloc[dev]);class_one=list(model.classes_).index(1)
        model_dir=root/'checkpoints/biomarkers';model_dir.mkdir(parents=True,exist_ok=True)
        background_rows=np.random.default_rng(SEED).choice(len(dev),size=min(32,len(dev)),replace=False)
        background_raw=X.iloc[dev].iloc[background_rows]
        background=model.named_steps['imputer'].transform(background_raw)
        model_path=model_dir/source/target_name/'selected_gbdt_stacking'/'v1'/'model.joblib';model_path.parent.mkdir(parents=True,exist_ok=True)
        joblib.dump({'model':model,'target':target_name,'feature_names':list(X.columns),'selected_model':best_name,
                     'background_features':background},model_path)
        model_sha256=file_sha256(model_path)
        for part,rows in [('calibration',cal),('test',test)]:
            prob=model.predict_proba(X.iloc[rows])[:,class_one]
            prediction_rows.extend({'record_id':ids[i],'target':target_name,'y_true':int(y.iloc[i]),
                'y_probability':float(p),'dataset':source,'selected_model':best_name,'split':part,
                'base_model_sha256':model_sha256} for i,p in zip(rows,prob,strict=True))
            split_rows.extend({'record_id':ids[i],'dataset':source,'target':target_name,'split':part} for i in rows)
        split_rows.extend({'record_id':ids[i],'dataset':source,'target':target_name,'split':'development'} for i in dev)
        print(f'{source}/{target_name}: {best_name}, development CV AUROC={best_score:.3f}, cal n={len(cal)}, test n={len(test)}')
predictions=pd.DataFrame(prediction_rows)
for source in DATASETS:
    folder=OUT/source;folder.mkdir(parents=True,exist_ok=True)
    subset=predictions[predictions.dataset==source]
    for split,filename in [('calibration','validation_predictions.csv'),('test','test_predictions.csv')]:
        subset.loc[subset.split==split].to_csv(folder/filename,index=False)
pd.DataFrame(split_rows).to_csv(OUT/'split_manifest.csv',index=False)
pd.DataFrame(cv_rows).to_csv(OUT/'development_cv_model_comparison.csv',index=False)
print('Saved independent predictions under',OUT)

zheen/acute_mi: HistGradientBoosting, development CV AUROC=0.997, cal n=198, test n=198
uci_heart_failure/hf_mortality: LogisticRegression, development CV AUROC=0.791, cal n=45, test n=45
framingham/ten_year_chd_risk: LogisticRegression, development CV AUROC=0.738, cal n=636, test n=636
mi_complications/fibr_preds: RandomForest, development CV AUROC=0.724, cal n=255, test n=255
mi_complications/preds_tah: LogisticRegression, development CV AUROC=0.417, cal n=255, test n=255
mi_complications/jelud_tah: LogisticRegression, development CV AUROC=0.786, cal n=255, test n=255
mi_complications/fibr_jelud: RandomForest, development CV AUROC=0.790, cal n=255, test n=255
mi_complications/a_v_blok: RandomForest, development CV AUROC=0.791, cal n=255, test n=255
mi_complications/o_l_post: RandomForest, development CV AUROC=0.779, cal n=253, test n=254
mi_complications/razriv: RandomForest, development CV AUROC=0.794, cal n=255, test n=255
mi_complications/dressler: LogisticRegression, development 